# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the mass spectrometry proteomic dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via its [FAIR2 Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) URL.

In [ ]:
# Ensure `mlcroissant` and visualization libraries are installed
!pip install mlcroissant
!pip install matplotlib seaborn

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset using Croissant
dataset = mlc.Dataset(croissant_url)
# Metadata access (see note: it's not a dict, but has attribute access)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}\n")
print(f"Version: {metadata.version}\n")
print(f"Published: {metadata.datePublished}\n")
print(f"License: {metadata.license}\n")

## 2. Data Overview
Review the available Record Sets, Fields, and their `@id` values.

> All entities are referenced by their `@id` (unique identifier in Croissant).

In [ ]:
# List all record sets, their fields, and columns by their @id
print("## Available Record Sets\n------------------------")
record_sets = list(dataset.record_sets)
for rset in record_sets:
    print(f'Record Set: @id={rset.id}, Name={rset.name}')

    print('  Fields:')
    for field in rset.fields:
        print(f'    - Field: @id={field.id}, Name={field.name}')
        # If this field is bound to columns (for tabular), list them
        if hasattr(field, 'columns') and field.columns:
            for col in field.columns:
                print(f'      > Column: @id={col.id}, Title={getattr(col, "name", getattr(col, "title", ""))}')
    print()

# Optionally: print the first N records from each record set
for rset in record_sets:
    print(f"First 2 records of Record Set: @id={rset.id}")
    for i, rec in enumerate(dataset.records(record_set=rset.id)):
        print(json.dumps(rec, indent=2))
        if i == 1:
            break
    print()


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We extract all main record sets and inspect their columns.

In [ ]:
# Extract data from all record sets into pandas DataFrames.
dataframes = {}
record_set_ids = [rset.id for rset in dataset.record_sets]

for record_set_id in record_set_ids:
    # Fetch and convert to DataFrame
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for Record Set: {record_set_id}")
        print(f"Columns (@id): {df.columns.tolist()}")
        print(df.head(2))
        print()
    else:
        print(f"No records found for Record Set: {record_set_id}")

# As a demonstration, let's pick the main table for further analysis
main_record_set = next(iter(dataframes.keys())) # Choose the first non-empty
print(f"Main record set for EDA: {main_record_set}")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps, such as filtering records, normalizing numeric fields, and grouping data for summary statistics.

We select a numeric field and a categorical/grouping field identified via their `@id`.

In [ ]:
# Choose a DataFrame from a key record set (@id must match that of overview)
df = dataframes[main_record_set]

# Example: Choose relevant @id fields (update these according to previous section output)
# Let's auto-detect a likely numeric field and a group field for demo
numeric_field = None
group_field = None

# Try to auto-detect: look for float columns
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field = col
        break
# Try to auto-detect group field as first string/categorical
for col in df.columns:
    if numeric_field and col != numeric_field and pd.api.types.is_string_dtype(df[col]):
        group_field = col
        break

if numeric_field:
    print(f"Using numeric field: {numeric_field}")
else:
    raise RuntimeError("No numeric field detected. Please edit numeric_field with a valid column '@id'.")

if group_field:
    print(f"Using group field: {group_field}\n")
else:
    print("No group field detected. Group-by operations will be skipped.\n")

# Filtering: Threshold at mean+stddev for demo
threshold = df[numeric_field].mean() + df[numeric_field].std()
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records where {numeric_field} > {threshold:.2f} (n={len(filtered_df)}):")
print(filtered_df.head(5))

# Normalization
filtered_df = filtered_df.copy()
filtered_df[f"{numeric_field}_normalized"] = (
    (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
)
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head(5))

# Group-By (if group_field is detected)
if group_field and group_field in df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nGrouped (mean) of {numeric_field} by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize distributions and relationships between numeric and category fields using Matplotlib and Seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Frequency")
plt.show()

# Boxplot by group, if group_field is available
if group_field and group_field in df.columns:
    plt.figure(figsize=(10,4))
    top_groups = df[group_field].value_counts().index[:6]
    sns.boxplot(data=df[df[group_field].isin(top_groups)], x=group_field, y=numeric_field)
    plt.title(f"{numeric_field} by {group_field} (top 6 groups)")
    plt.ylabel(numeric_field)
    plt.xlabel(group_field)
    plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the `mlcroissant` library.

We reviewed the structure via `@id` fields, extracted data, performed filtering, normalization, grouped analysis, and visualized distributions.

You can now proceed with deeper analyses, such as machine learning, domain-specific statistics, or custom visualizations, guided by the FAIR and explicit structure of this dataset.